# 4장. pandas로 데이터에 질문하기

이 노트북은 데이터를 선택·필터링·정렬하고, 안전하게 병합한 뒤 완료 주문 기준 요약표를 만드는 실습 자료입니다.


## 학습 목표

- 실제 컬럼명과 값의 종류를 확인한 뒤 필터링합니다.
- 수량과 단가로 주문 상세 금액을 계산합니다.
- `validate`와 `indicator`를 사용해 병합 결과를 검증합니다.
- 취소·환불 주문을 제외하고 완료 주문 기준 매출을 계산합니다.
- 개인정보를 최소화한 분석 결과를 저장합니다.


## 1. 프로젝트 루트와 실행 환경 확인

VS Code에서 Notebook을 실행하면 현재 작업 폴더가 프로젝트 루트 또는 `notebooks` 폴더일 수 있습니다. 아래 코드는 상위 폴더를 확인해 프로젝트 루트를 찾습니다.


In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print('Python 실행 파일:', sys.executable)
print('현재 작업 폴더:', Path.cwd())
print('프로젝트 루트:', PROJECT_ROOT)
print('데이터 폴더:', DATA_DIR)
print('결과 저장 폴더:', REPORT_DIR)


Python 실행 파일: /workspaces/llm-data-analysis-study/llm-data-analysis-course/.venv/bin/python
현재 작업 폴더: /workspaces/llm-data-analysis-study/llm-data-analysis-course/notebooks
프로젝트 루트: /workspaces/llm-data-analysis-study/llm-data-analysis-course
데이터 폴더: /workspaces/llm-data-analysis-study/llm-data-analysis-course/data/raw
결과 저장 폴더: /workspaces/llm-data-analysis-study/llm-data-analysis-course/reports


## 2. 데이터 파일 확인과 불러오기

파일이 없다면 프로젝트 루트에서 `python scripts/generate_sample_data.py`를 먼저 실행하세요.


In [2]:
required_files = ['customers.csv', 'products.csv', 'orders.csv', 'order_items.csv']
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]

if missing_files:
    raise FileNotFoundError(
        '필요한 데이터 파일이 없습니다: ' + ', '.join(missing_files)
        + '. 프로젝트 루트에서 python scripts/generate_sample_data.py를 실행하세요.'
    )

customers = pd.read_csv(DATA_DIR / 'customers.csv')
products = pd.read_csv(DATA_DIR / 'products.csv')
orders = pd.read_csv(DATA_DIR / 'orders.csv')
order_items = pd.read_csv(DATA_DIR / 'order_items.csv')

print('데이터 불러오기 완료')


데이터 불러오기 완료


## 3. 데이터 구조와 필수 컬럼 확인

LLM이 작성한 코드가 실제 컬럼명과 일치하는지 먼저 확인합니다.


In [3]:
datasets = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

expected_columns = {
    'customers': ['customer_id', 'gender', 'age', 'city'],
    'products': ['product_id', 'product_name', 'category', 'price'],
    'orders': ['order_id', 'customer_id', 'order_date', 'order_status'],
    'order_items': ['order_id', 'product_id', 'quantity', 'unit_price'],
}

for name, df in datasets.items():
    missing = [col for col in expected_columns[name] if col not in df.columns]
    print(name, df.shape, df.columns.tolist())
    if missing:
        raise KeyError(f'{name}에 필요한 컬럼이 없습니다: {missing}')


customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
orders (300, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items (764, 5) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']


## 4. 컬럼 선택과 행 필터링

이 저장소의 샘플 데이터는 도시명을 `서울`, `부산`처럼 한글로 생성합니다. 필터링 전에 실제 값을 확인합니다.


In [4]:
customer_basic = customers[['customer_id', 'gender', 'age', 'city']]
display(customer_basic.head())

print('도시별 고객 수')
display(customers['city'].value_counts())

customers_over_30 = customers[customers['age'] >= 30]
seoul_customers = customers[customers['city'] == '서울']
city_customers = customers[customers['city'].isin(['서울', '부산'])]

print('30세 이상 고객 수:', len(customers_over_30))
print('서울 고객 수:', len(seoul_customers))
print('서울 또는 부산 고객 수:', len(city_customers))


,customer_id,gender,age,city
0,1,F,19,광주
1,2,F,32,대구
2,3,F,61,성남
3,4,F,55,울산
4,5,F,19,부산


도시별 고객 수


city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

30세 이상 고객 수: 111
서울 고객 수: 15
서울 또는 부산 고객 수: 31


## 5. 정렬과 주문 상태 확인


In [5]:
display(products.sort_values('price', ascending=False).head(10))
display(customers.sort_values('age', ascending=False).head(10))

print('주문 상태별 건수')
display(orders['order_status'].value_counts())


,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


,customer_id,name,gender,age,city,signup_date
14,15,장정식,M,69,서울,2026-06-04
8,9,송지민,M,69,서울,2025-10-19
54,55,윤영철,M,69,광주,2023-07-31
78,79,이서준,M,69,부산,2023-07-14
136,137,김선영,M,68,울산,2024-11-28
45,46,김서현,M,67,대전,2026-05-13
132,133,김성진,M,67,성남,2026-05-29
123,124,김시우,M,67,대구,2024-09-24
11,12,김정남,F,66,대전,2024-09-30
57,58,김성훈,F,66,성남,2025-06-14


주문 상태별 건수


order_status
completed    184
cancelled     64
refunded      52
Name: count, dtype: int64

## 6. 주문 상세 금액 만들기

`line_total`은 주문 상세 1행의 금액입니다. 주문 상태를 연결하기 전 합계는 확정 매출이 아니라 전체 주문 상세 금액입니다.


In [6]:
order_items = order_items.copy()
order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

all_order_amount = order_items['line_total'].sum()
print('전체 주문 상세 금액:', all_order_amount)
display(order_items[['order_id', 'product_id', 'quantity', 'unit_price', 'line_total']].head())


전체 주문 상세 금액: 255610000


,order_id,product_id,quantity,unit_price,line_total
0,1,100,3,102000,306000
1,1,87,5,25000,125000
2,1,7,3,142000,426000
3,1,9,3,193000,579000
4,2,72,4,189000,756000


## 7. 주문 데이터 병합과 검증

`validate='many_to_one'`은 주문 상세의 동일 주문 ID는 여러 번 나올 수 있지만 주문 테이블의 주문 ID는 한 번만 나와야 한다는 뜻입니다.


In [7]:
print('orders.order_id 중복 수:', orders['order_id'].duplicated().sum())

order_sales = order_items.merge(
    orders[['order_id', 'customer_id', 'order_date', 'order_status']],
    on='order_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(order_items))
print('병합 후 행 수:', len(order_sales))
display(order_sales['_merge'].value_counts())

order_sales = order_sales.drop(columns='_merge')


orders.order_id 중복 수: 0
병합 전 행 수: 764
병합 후 행 수: 764


_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

## 8. 완료 주문만 선택하기

이후의 매출 요약은 `order_status == 'completed'`인 주문만 사용합니다.


In [8]:
order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
print('날짜 변환 실패:', order_sales['order_date'].isna().sum())

completed_order_sales = order_sales[
    order_sales['order_status'] == 'completed'
].copy()

completed_order_sales['order_month'] = (
    completed_order_sales['order_date'].dt.to_period('M').astype(str)
)

print('전체 주문 상세 행 수:', len(order_sales))
print('완료 주문 상세 행 수:', len(completed_order_sales))
print('완료 주문 매출:', completed_order_sales['line_total'].sum())


날짜 변환 실패: 0
전체 주문 상세 행 수: 764
완료 주문 상세 행 수: 474
완료 주문 매출: 148990000


## 9. 상품 데이터 병합과 검증


In [9]:
print('products.product_id 중복 수:', products['product_id'].duplicated().sum())

completed_sales_items = completed_order_sales.merge(
    products,
    on='product_id',
    how='left',
    validate='many_to_one',
    indicator=True,
)

print('병합 전 행 수:', len(completed_order_sales))
print('병합 후 행 수:', len(completed_sales_items))
display(completed_sales_items['_merge'].value_counts())

completed_sales_items = completed_sales_items.drop(columns='_merge')


products.product_id 중복 수: 0
병합 전 행 수: 474
병합 후 행 수: 474


_merge
both          474
left_only       0
right_only      0
Name: count, dtype: int64

## 10. 카테고리별·상품별 매출


In [10]:
category_sales = (
    completed_sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

display(category_sales)

product_sales = (
    completed_sales_items
    .groupby(['product_id', 'product_name', 'category'], as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

display(product_sales.head(10))


,category,total_quantity,total_sales,sales_ratio
3,스포츠,295,31743000,21.31
5,전자기기,259,26400000,17.72
2,생활용품,272,23915000,16.05
1,뷰티,223,23383000,15.69
4,식품,133,16573000,11.12
0,도서,149,16389000,11.00
6,패션,111,10587000,7.11


,product_id,product_name,category,total_quantity,total_sales
39,41,스포츠 상품 041,스포츠,35,5705000
11,12,식품 상품 012,식품,25,4375000
8,9,스포츠 상품 009,스포츠,20,3860000
70,72,뷰티 상품 072,뷰티,20,3780000
69,71,전자기기 상품 071,전자기기,23,3703000
66,68,스포츠 상품 068,스포츠,26,3640000
78,81,전자기기 상품 081,전자기기,22,3630000
10,11,패션 상품 011,패션,31,3565000
20,22,생활용품 상품 022,생활용품,29,3248000
86,89,생활용품 상품 089,생활용품,30,3090000


## 11. 월별 매출


In [11]:
monthly_summary = (
    completed_order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_summary['average_order_value'] = (
    monthly_summary['total_sales'] / monthly_summary['order_count']
).round(0)

display(monthly_summary)


,order_month,total_sales,order_count,average_order_value
0,2025-07,5869000,8,733625.0
1,2025-08,15621000,18,867833.0
2,2025-09,10190000,13,783846.0
3,2025-10,25766000,26,991000.0
4,2025-11,8812000,12,734333.0
5,2025-12,11501000,14,821500.0
6,2026-01,17423000,22,791955.0
7,2026-02,9749000,17,573471.0
8,2026-03,13429000,14,959214.0
9,2026-04,17553000,23,763174.0


## 12. 고객별 구매 금액

고객 ID로 먼저 집계한 뒤 고객 속성을 붙입니다. 출력에는 실제 이름 대신 익명화된 고객 라벨을 사용합니다.


In [12]:
customer_sales = (
    completed_order_sales
    .groupby('customer_id', as_index=False)
    .agg(
        order_count=('order_id', 'nunique'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

customer_sales = customer_sales.merge(
    customers[['customer_id', 'city']],
    on='customer_id',
    how='left',
    validate='one_to_one',
)

customer_sales['customer_label'] = 'Customer ' + customer_sales['customer_id'].astype(str)
display(customer_sales[['customer_label', 'city', 'order_count', 'total_sales']].head(10))


,customer_label,city,order_count,total_sales
0,Customer 117,성남,5,4100000
1,Customer 102,고양,4,3996000
2,Customer 83,수원,4,3880000
3,Customer 30,서울,5,3590000
4,Customer 40,서울,4,3523000
5,Customer 20,인천,2,3191000
6,Customer 3,성남,2,3178000
7,Customer 111,광주,3,3153000
8,Customer 66,서울,4,3093000
9,Customer 147,부산,2,2990000


## 13. 결과 저장하기


In [ ]:
outputs = {
    'ch04_category_sales.csv': category_sales,
    'ch04_product_sales.csv': product_sales,
    'ch04_monthly_sales.csv': monthly_summary,
    'ch04_customer_sales.csv': customer_sales,
}

for filename, df in outputs.items():
    path = REPORT_DIR / filename
    df.to_csv(path, index=False, encoding='utf-8-sig')
    print(filename, path.exists(), path.stat().st_size)


ch04_category_sales.csv True 246
ch04_product_sales.csv True 4310
ch04_monthly_sales.csv True 428
ch04_customer_sales.csv True 3286


## 14. LLM 코드 검증 연습

```text
LLM이 다음 코드를 제안했습니다.

category_sales = order_items.groupby('category')['line_total'].sum()

현재 데이터에서 이 코드가 바로 실행 가능한지 검토해 주세요.
category 컬럼의 위치, 주문 상태 필터링, merge validate와 indicator를 포함해
안전한 수정 코드와 검증 순서를 설명해 주세요.
```


해당 코드는 현재 데이터에서 바로 실행 가능한 코드가 아닙니다.
먼저 category 컬럼은 order_items에 있는 컬럼이 아니라 products 데이터에 있는 컬럼입니다. 
따라서 order_items만으로는 카테고리별 집계를 할 수 없습니다. 
또한 order_items에는 주문 상태를 나타내는 order_status 컬럼도 없기 때문에 completed 주문만 필터링할 수 없습니다. 
line_total 역시 미리 만들어져 있지 않다면 quantity * unit_price로 먼저 생성해야 합니다.
결국 안전한 분석을 위해서는 order_items에 line_total을 생성한 뒤, orders를 병합하여 주문 상태를 가져오고, products를 병합하여 카테고리 정보를 가져와야 합니다. 이때 병합 key가 올바른지, 오른쪽 테이블 key가 중복되지 않았는지, 병합 후 미매칭 데이터가 없는지 확인해야 합니다.

즉, 검증 순서를 정리해보면 아래와 같습니다.

1. 실제 컬럼명을 확인한다.
2. `category` 컬럼이 `order_items`가 아니라 `products`에 있는지 확인한다.
3. `line_total = quantity * unit_price`를 생성한다.
4. `orders.order_id` 중복 여부를 확인한다.
5. `order_items`와 `orders`를 `order_id` 기준으로 병합하고 `validate="many_to_one"`으로 관계를 검증한다.
6. `indicator=True`로 미매칭 데이터가 있는지 확인한다.
7. `products.product_id` 중복 여부를 확인한다.
8. `products`를 `product_id` 기준으로 병합하고 `validate="many_to_one"`, `indicator=True`로 검증한다.
9. `order_status == "completed"`만 필터링한다.
10. 카테고리별 `total_sales`, `total_quantity`, `order_count`를 집계한다.
11. 원본 completed `line_total` 합계와 카테고리별 합계가 일치하는지 확인한다.
12. 총합이 일치하는지 확인한다.

In [16]:
# line_total 생성
order_items = order_items.copy()
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

# orders key 중복 확인
print("orders.order_id 중복 수:", orders["order_id"].duplicated().sum())

# order_items + orders 병합
merged = order_items.merge(
    orders,
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("order_items + orders 병합 결과")
print(merged["_merge"].value_counts())

print("병합 전 order_items 행 수:", len(order_items))
print("병합 후 merged 행 수:", len(merged))

# 미매칭 확인
unmatched_orders = merged[merged["_merge"] != "both"]
print("orders 미매칭 행 수:", len(unmatched_orders))

merged = merged.drop(columns="_merge")

# products key 중복 확인
print("products.product_id 중복 수:", products["product_id"].duplicated().sum())

# products 병합
merged = merged.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("products 병합 결과")
print(merged["_merge"].value_counts())

# 미매칭 확인
unmatched_products = merged[merged["_merge"] != "both"]
print("products 미매칭 행 수:", len(unmatched_products))

merged = merged.drop(columns="_merge")

# completed 주문만 필터링
completed = merged[merged["order_status"] == "completed"].copy()

# 카테고리별 집계
category_sales = (
    completed
    .groupby("category", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        total_quantity=("quantity", "sum"),
        order_count=("order_id", "nunique")
    )
    .sort_values("total_sales", ascending=False)
)

category_sales

orders.order_id 중복 수: 0
order_items + orders 병합 결과
_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64
병합 전 order_items 행 수: 764
병합 후 merged 행 수: 764
orders 미매칭 행 수: 0
products.product_id 중복 수: 0
products 병합 결과
_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64
products 미매칭 행 수: 0


,category,total_sales,total_quantity,order_count
3,스포츠,31743000,295,85
5,전자기기,26400000,259,60
2,생활용품,23915000,272,65
1,뷰티,23383000,223,65
4,식품,16573000,133,36
0,도서,16389000,149,52
6,패션,10587000,111,33


In [17]:
base_total = completed["line_total"].sum()
category_total = category_sales["total_sales"].sum()

print("원본 completed line_total 합계:", base_total)
print("카테고리별 total_sales 합계:", category_total)
print("일치 여부:", base_total == category_total)

원본 completed line_total 합계: 148990000
카테고리별 total_sales 합계: 148990000
일치 여부: True


## (추가) 총합 일치 검증 상세

In [18]:
completed_total = completed_order_sales["line_total"].sum()

total_check = pd.DataFrame([
    {
        "집계 기준": "원본 completed line_total",
        "total_sales": completed_total,
    },
    {
        "집계 기준": "카테고리별 total_sales 합계",
        "total_sales": category_sales["total_sales"].sum(),
    },
    {
        "집계 기준": "상품별 total_sales 합계",
        "total_sales": product_sales["total_sales"].sum(),
    },
    {
        "집계 기준": "월별 total_sales 합계",
        "total_sales": monthly_summary["total_sales"].sum(),
    },
    {
        "집계 기준": "고객별 total_sales 합계",
        "total_sales": customer_sales["total_sales"].sum(),
    },
])

total_check["원본과의 차이"] = total_check["total_sales"] - completed_total
total_check["일치 여부"] = total_check["원본과의 차이"].eq(0)

display(total_check)

assert total_check["일치 여부"].all(), "집계별 total_sales 합계가 일치하지 않습니다."
print("검증 완료: 모든 집계의 total_sales 합계가 일치합니다.")

,집계 기준,total_sales,원본과의 차이,일치 여부
0,원본 completed line_total,148990000,0,True
1,카테고리별 total_sales 합계,148990000,0,True
2,상품별 total_sales 합계,148990000,0,True
3,월별 total_sales 합계,148990000,0,True
4,고객별 total_sales 합계,148990000,0,True


검증 완료: 모든 집계의 total_sales 합계가 일치합니다.


| 검증 단계 | 실제 확인된 결과 | 나의 판단 |
|---|---:|---|
| 실제 컬럼명 확인 | `category`는 `order_items`가 아니라 `products`에 있음 | `order_items`만으로는 카테고리별 집계를 할 수 없으므로 `products` 병합이 필요하다고 판단했습니다. |
| 주문 상태 컬럼 확인 | `order_status`는 `orders`에 있음 | completed 주문만 분석하려면 `orders`를 `order_id` 기준으로 병합해야 한다고 판단했습니다. |
| 주문 상태 실제 값 확인 | `completed`, `cancelled`, `refunded` | 과제 조건에 맞게 `order_status == "completed"`만 포함해야 한다고 판단했습니다. |
| 금액 컬럼 확인 | `line_total = quantity × unit_price`로 생성 | 주문 상세 단위 금액을 직접 계산한 뒤 집계해야 한다고 판단했습니다. |
| `orders.order_id` 중복 확인 | 중복 수 `0` | `orders`는 `order_id` 기준으로 고유하므로 `order_items`와 `many_to_one` 병합이 가능하다고 판단했습니다. |
| `products.product_id` 중복 확인 | 중복 수 `0` | 상품 정보도 `product_id` 기준으로 고유하므로 상품 병합 시 행이 불필요하게 늘어나지 않는다고 판단했습니다. |
| `order_items`와 `orders` 병합 검증 | `validate="many_to_one"` 통과, 미매칭 `0건` | 주문 상세 데이터가 주문 정보와 정상적으로 연결되었으므로 병합이 안전하다고 판단했습니다. |
| 병합 전후 행 수 확인 | 병합 후에도 주문 상세 행 수 유지 | merge 과정에서 주문 상세 행이 중복되거나 누락되지 않았다고 판단했습니다. |
| completed 필터 적용 | completed 주문 상세 행 수 `474건` | 이후 모든 집계는 같은 completed 주문 범위에서 수행해야 한다고 판단했습니다. |
| completed 원본 합계 확인 | `148,990,000원` | 이 값을 기준 합계로 두고 다른 집계 결과와 비교해야 한다고 판단했습니다. |
| 카테고리별 집계 결과 | 1위 `스포츠`, `31,743,000원` | completed 주문 기준 금액이 가장 큰 카테고리는 스포츠라고 판단했습니다. |
| 상품별 집계 결과 | 1위 `스포츠 상품 041`, `5,705,000원` | 스포츠 카테고리 안에서도 특정 상품이 금액에 크게 기여했는지 추가 확인할 필요가 있다고 판단했습니다. |
| 월별 집계 결과 | 1위 `2025-10`, `25,766,000원` | 특정 월에 completed 주문 금액이 높게 나타났으므로 시즌성이나 이벤트 가능성을 추가로 볼 수 있다고 판단했습니다. |
| 고객별 집계 결과 | 1위 고객 ID `117`, `4,100,000원` | 특정 고객의 구매 금액이 높게 나타나므로 고객별 편중 여부를 추가로 확인할 수 있다고 판단했습니다. |
| 총합 일치 검증 | 원본, 카테고리별, 상품별, 월별, 고객별 합계 모두 `148,990,000원` | 같은 completed 주문 범위가 끝까지 유지되었고, 집계 과정에서 누락이나 중복이 없다고 판단했습니다. |
| LLM 코드 최종 판단 | 원래 제안 코드 그대로는 사용 불가 | 실제 컬럼 위치, completed 필터, merge 검증, 총합 검증을 반영해 수정 후 사용하는 것이 적절하다고 판단했습니다. |

## 15. 실습 과제

1. 40세 이상 고객을 추출합니다.
2. 상품 가격이 낮은 순서대로 10개를 출력합니다.
3. 결제수단별 주문 수와 비율을 계산합니다.
4. 카테고리별 평균 상품 가격을 계산합니다.
5. 완료 주문과 전체 주문의 금액 차이를 계산합니다.
6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트를 작성합니다.


In [20]:
# 과제 코드를 아래에 작성하세요.

# 실습 전 컬럼 확인
print("customers columns:", customers.columns.tolist())
print("products columns:", products.columns.tolist())
print("orders columns:", orders.columns.tolist())
print("order_items columns:", order_items.columns.tolist())

# 1. 40세 이상 고객 추출
customers_40_over = customers[customers["age"] >= 40].copy()

print("\n1. 40세 이상 고객")
display(customers_40_over.head())

# 2. 상품 가격이 낮은 순서대로 10개 출력
low_price_products = (
    products
    .sort_values("price", ascending=True)
    .head(10)
)

print("\n2. 상품 가격이 낮은 순서대로 10개")
display(low_price_products)

# 3. 결제수단별 주문 수와 비율 계산
payment_summary = (
    orders
    .groupby("payment_method", as_index=False)
    .agg(order_count=("order_id", "nunique"))
)

payment_summary["order_ratio"] = (
    payment_summary["order_count"] / payment_summary["order_count"].sum()
)

payment_summary["order_ratio_percent"] = (
    payment_summary["order_ratio"] * 100
).round(2)

payment_summary = payment_summary.sort_values("order_count", ascending=False)

print("\n3. 결제수단별 주문 수와 비율")
display(payment_summary)

# 4. 카테고리별 평균 상품 가격 계산
category_avg_price = (
    products
    .groupby("category", as_index=False)
    .agg(avg_price=("price", "mean"))
    .sort_values("avg_price", ascending=False)
)

print("\n4. 카테고리별 평균 상품 가격")
display(category_avg_price)

# 5. 완료 주문과 전체 주문의 금액 차이 계산

# line_total이 없으면 생성
if "line_total" not in order_items.columns:
    order_items = order_items.copy()
    order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

# 주문 정보 병합
order_amounts = order_items.merge(
    orders,
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

print("\n5. 주문 정보 병합 검증")
print(order_amounts["_merge"].value_counts())

order_amounts = order_amounts.drop(columns="_merge")

# 전체 주문 금액
all_order_total = order_amounts["line_total"].sum()

# 완료 주문 금액
completed_order_total = order_amounts.loc[
    order_amounts["order_status"] == "completed",
    "line_total"
].sum()

amount_diff = all_order_total - completed_order_total

amount_compare = pd.DataFrame({
    "구분": ["전체 주문 금액", "완료 주문 금액", "차이"],
    "금액": [all_order_total, completed_order_total, amount_diff]
})

print("\n5. 완료 주문과 전체 주문의 금액 차이")
display(amount_compare)

customers columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
products columns: ['product_id', 'product_name', 'category', 'price']
orders columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
order_items columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']

1. 40세 이상 고객


,customer_id,name,gender,age,city,signup_date
2,3,이경수,F,61,성남,2024-06-12
3,4,조영호,F,55,울산,2026-04-13
6,7,이상현,F,53,인천,2024-12-12
8,9,송지민,M,69,서울,2025-10-19
9,10,유도현,F,62,울산,2024-10-06



2. 상품 가격이 낮은 순서대로 10개


,product_id,product_name,category,price
5,6,전자기기 상품 006,전자기기,5000
27,28,뷰티 상품 028,뷰티,5000
97,98,스포츠 상품 098,스포츠,10000
46,47,생활용품 상품 047,생활용품,11000
94,95,전자기기 상품 095,전자기기,20000
49,50,뷰티 상품 050,뷰티,23000
82,83,전자기기 상품 083,전자기기,24000
86,87,도서 상품 087,도서,25000
41,42,패션 상품 042,패션,28000
58,59,뷰티 상품 059,뷰티,32000



3. 결제수단별 주문 수와 비율


,payment_method,order_count,order_ratio,order_ratio_percent
2,kakao_pay,79,0.263333,26.33
3,naver_pay,77,0.256667,25.67
0,bank_transfer,74,0.246667,24.67
1,card,70,0.233333,23.33



4. 카테고리별 평균 상품 가격


,category,avg_price
4,식품,137142.857143
1,뷰티,117687.500000
6,패션,115909.090909
3,스포츠,111578.947368
0,도서,106857.142857
5,전자기기,101588.235294
2,생활용품,96437.500000



5. 주문 정보 병합 검증
_merge
both          764
left_only       0
right_only      0
Name: count, dtype: int64

5. 완료 주문과 전체 주문의 금액 차이


,구분,금액
0,전체 주문 금액,255610000
1,완료 주문 금액,148990000
2,차이,106620000


### 6. 병합 검증을 포함한 고객별 구매 금액 코드 요청 프롬프트

```text
고객별 completed 주문 기준 구매 금액을 계산하는 pandas 코드를 작성해줘.

사용 데이터:
- customers.csv
- orders.csv
- order_items.csv

분석 조건:
- 주문 상태는 order_status == "completed"만 포함해줘.
- 주문 상세 금액은 line_total = quantity * unit_price로 계산해줘.
- order_items와 orders는 order_id 기준으로 병합해줘.
- orders와 customers는 customer_id 기준으로 병합해줘.
- 병합 전 key 중복 여부를 확인해줘.
- 병합 시 validate="many_to_one"을 사용해 merge 관계를 검증해줘.
- indicator=True를 사용해 미매칭 데이터가 있는지 확인해줘.
- 병합 전후 행 수를 비교해줘.
- 고객별 total_sales, total_quantity, order_count를 계산해줘.
- 주문 수는 order_id.nunique()로 계산해줘.
- 원본 completed line_total 합계와 고객별 total_sales 합계가 일치하는지 검증해줘.

주의:
코드가 실행되는 것보다 실제 컬럼명, completed 필터 조건, merge 관계, 미매칭 여부, 총합 일치 검증이 올바른지가 중요해.

## 정리

이번 장에서는 실제 값 확인, 컬럼 선택, 필터링, 정렬, 파생 컬럼, 병합 검증, 완료 주문 기준 집계, CSV 저장 과정을 수행했습니다. 다음 장에서는 결측치, 중복, 타입 오류, 날짜 형식, 이상값 후보를 다루는 데이터 전처리로 이어집니다.


# Chapter 04 제출 답안 양식. pandas로 데이터에 질문하기

> 주 제출물은 실행 완료 Notebook `chapter04/chapter04.ipynb`입니다. 이 양식의 항목을 Notebook의 Markdown 셀로 추가해 작성합니다.

## 0. 제출 정보
- 이름: `이경석`
- GitHub ID: `leeks2026`
- 작성일: `20260928`
- 최종 제출 URL: `https://github.com/leeks2026/llm-data-analysis-study/blob/main/chapter04/chapter04.ipynb`

## 1. 질문과 필요한 데이터 선택
### 내가 확인하려는 질문
어떤 상품 카테고리의 completed 주문 기준 금액이 가장 큰가?

### 사용한 파일/컬럼
이번 분석의 질문에 맞춰 주문 상태, 주문 상세 금액, 상품 카테고리 정보를 함께 사용하였습니다.
사용한 파일과 주요 컬럼은 아래와 같습니다.

- `orders.csv`
  - `order_id`: 주문 상세 데이터와 병합하기 위한 주문 key
  - `customer_id`: 고객별 집계에 사용
  - `order_date`: 월별 집계를 위한 날짜 컬럼
  - `order_status`: 분석에 포함할 주문 상태를 구분하는 컬럼

- `order_items.csv`
  - `order_id`: 주문 데이터와 병합하기 위한 주문 key
  - `product_id`: 상품 데이터와 병합하기 위한 상품 key
  - `quantity`: 주문 수량
  - `unit_price`: 상품 단가
  - `line_total`: `quantity × unit_price`로 계산한 주문 상세 금액

- `products.csv`
  - `product_id`: 주문 상세 데이터와 병합하기 위한 상품 key
  - `product_name`: 상품별 집계에 사용
  - `category`: 최종 질문인 카테고리별 집계 기준

- `customers.csv`
  - `customer_id`: 주문 데이터와 병합하기 위한 고객 key
  - 고객 관련 컬럼: 고객별 구매 금액을 집계할 때 사용

분석에 포함한 주문 상태는 `order_status == "completed"`입니다.  
취소(`cancelled`) 또는 환불(`refunded`) 주문은 completed 주문 기준 금액을 왜곡할 수 있으므로 제외했습니다.

금액 계산식은 다음과 같습니다.
line_total = quantity × unit_price

### 결과 관찰
실제 데이터에서 `order_status` 값은 `completed`, `cancelled`, `refunded`로 확인되었습니다. 
이 중 completed 주문만 사용했을 때 주문 상세 기준 합계는 `148,990,000원`이었으며
카테고리별 집계 결과에서는 `스포츠` 카테고리의 `total_sales`가 `31,743,000원`으로 가장 컸습니다.

### 나의 해석과 판단
질문이 “completed 주문 기준 금액”을 묻고 있으므로, 취소되었거나 환불된 주문은 분석 범위에서 제외해야 한다고 판단했습니다. 
또한 상품 카테고리별 금액을 비교하려면 주문 상세의 수량과 단가, 상품의 카테고리 정보, 주문의 상태 정보가 모두 필요하므로 `order_items`, `orders`, `products` 데이터를 함께 사용했습니다.

### 업무·분석적 의미
completed 주문 기준으로 어떤 카테고리에서 금액 기여가 큰지 확인하면 재고 관리, 프로모션 기획, 카테고리별 매출 목표 설정에 활용할 수 있습니다. 
특히 금액이 큰 카테고리는 주요 매출원일 가능성이 있으므로 더 세부적인 상품 구성과 고객 특성을 추가로 분석할 가치가 있습니다.

### 한계와 추가 확인 사항
이번 분석의 `total_sales`는 `quantity * unit_price`의 합계이며, 실제 회사의 순매출이나 이익을 의미하지는 않습니다. 
할인, 쿠폰, 배송비, 반품 처리, 세금, 원가 정보가 반영되지 않았기 때문입니다. 
따라서 금액이 가장 큰 카테고리가 반드시 가장 수익성이 높거나 가장 인기 있는 카테고리라고 단정할 수는 없습니다.

## 2. 필터링·정렬·파생 컬럼
- 적용한 필터 조건: `order_status == "completed"`
- 정렬 기준: `total_sales` 기준 내림차순
- 만든 파생 컬럼: `line_total`
- `line_total` 계산식: `quantity * unit_price`

![필터와 파생 컬럼](images/step02_transform.png)

### 결과 관찰
주문 상세 데이터에 `line_total` 컬럼을 추가하여 각 주문 라인의 금액을 계산했습니다. 
이후 completed 주문만 필터링해 분석에 사용했고 completed 주문 상세의 `line_total` 합계는 `148,990,000원`으로 확인되었습다.

### 나의 해석과 판단
이번 과제의 질문은 완료된 주문 기준의 금액을 비교하는 것이므로 completed 주문만 사용하는 것이 적절합니다. 
만약 cancelled나 refunded 주문을 포함하면 실제로 완료되지 않은 주문 금액까지 포함되어 카테고리별 결과가 왜곡될 수 있습니다.

### 업무·분석적 의미
필터 조건을 명확히 정의하면 분석 결과를 해석할 때 혼동을 줄일 수 있습니다. 
예를 들어 운영팀은 completed 주문 기준의 판매 흐름을 보고, CS팀은 cancelled나 refunded 주문을 별도로 분석하는 방식으로 목적에 따라 데이터를 나누어 볼 수 있습니다.

### 한계와 추가 확인 사항
completed 상태라고 해도 실제 정산 완료 여부와 완전히 같다고 볼 수는 없습니다. 
실제 매출 분석을 하려면 결제 완료 시점, 환불 처리 시점, 할인 금액, 세금, 배송비 등의 정보가 추가로 필요합니다.

## 3. merge 검증
- 병합한 데이터: `order_items`, `orders`, `products`, `customers`
- 사용한 key: `order_id`, `product_id`, `customer_id`
- `validate` 결과: `order_items`와 `orders` 병합은 `many_to_one` 관계로 검증
- `indicator` 결과: 미매칭 데이터 없이 모두 매칭됨
- 병합 전/후 행 수: 병합 후에도 주문 상세 기준 행 수가 유지됨

![merge 검증](images/step03_merge.png)

### 결과 관찰
`orders.order_id`에는 중복이 없었고, `order_items`와 `orders`를 `order_id` 기준으로 병합했을 때 `validate="many_to_one"` 조건을 만족했습니다. `_merge` 결과에서도 미매칭 데이터가 확인되지 않았습니다. 따라서 주문 상세 행이 잘못 늘어나거나 누락되지 않은 것으로 판단했습니다.

### 나의 해석과 판단
이번 병합은 하나의 주문에 여러 주문 상세가 연결되는 구조이므로 `many_to_one` 관계가 맞습니다. 
`orders` 쪽의 `order_id`가 중복되지 않았고, 병합 후 행 수가 유지되었으며, 
미매칭도 없었기 때문에 안전한 병합이라고 판단했습니다.

### 업무·분석적 의미
잘못된 merge는 금액 합계를 크게 왜곡할 수 있습니다. 
예를 들어 오른쪽 테이블의 key가 중복되어 있으면 주문 상세 행이 중복 생성되어 `total_sales`가 실제보다 커질 수 있습니다. 
반대로 미매칭이 있으면 일부 주문이 누락되어 결과가 작게 계산될 수 있습니다.

### 한계와 추가 확인 사항
key 값이 형식상 매칭되더라도 데이터 입력 오류가 완전히 없다고 단정할 수는 없습니다. 
실제 업무에서는 주문 생성 시점, 상품 상태, 고객 ID의 유효성 등도 함께 확인하는 것이 좋습니다.

## 4. completed 주문 범위와 집계
- 분석 범위 정의: `order_status == "completed"`인 주문의 주문 상세 데이터
- 카테고리별 결과: `스포츠` 카테고리의 `total_sales`가 `31,743,000원`으로 가장 큼
- 상품별 결과: `스포츠 상품 041`의 `total_sales`가 `5,705,000원`으로 가장 큼
- 월별 결과: `2025-10`의 `total_sales`가 `25,766,000원`으로 가장 큼
- 고객별 결과: 고객 ID `117`의 `total_sales`가 `4,100,000원`으로 가장 큼

![핵심 집계 결과](images/step04_groupby.png)

### 결과 관찰
completed 주문 기준으로 카테고리별 금액을 집계한 결과, `스포츠` 카테고리가 가장 높은 금액을 기록했습니다. 
상품별로는 `스포츠 상품 041`이 가장 높은 금액을 보였고, 월별로는 `2025-10`의 금액이 가장 컸습니다. 
고객별 집계에서는 고객 ID `117`의 구매 금액이 가장 높았습니다.

### 나의 해석과 판단
가장 중요한 결과는 `스포츠` 카테고리가 completed 주문 기준 금액 1위라는 점입니다. 
과제의 핵심 질문이 카테고리별 금액 비교이기 때문입니다. 
다만 상품별, 월별, 고객별 결과를 함께 보면 이 금액이 특정 상품, 특정 월, 특정 고객에 의해 크게 영향을 받았는지도 추가로 살펴볼 수 있습니다.

### 업무·분석적 의미
스포츠 카테고리가 금액 기준으로 가장 크다면 해당 카테고리의 재고 확보, 인기 상품 관리, 프로모션 기획을 우선 검토할 수 있습니다. 
또한 월별 변동이 크다면 시즌성이나 특정 이벤트의 영향을 분석해 마케팅 일정과 연결할 수 있습니다.


### 한계와 추가 확인 사항
`total_sales`는 completed 주문의 `quantity * unit_price` 합계일 뿐 회계상 순매출과 같다고 단정할 수 없습니다. 
할인, 쿠폰, 환불, 배송비, 세금, 원가가 반영되지 않았기 때문입니다. 
또한 금액이 높다는 것은 단가가 높아서일 수도 있고, 수량이 많아서일 수도 있으므로 인기도나 수익성을 판단하려면 추가 분석이 필요합니다.

## 5. 총합 일치 검증
- 원본 completed `line_total` 합계: `148,990,000원`
- 카테고리 합계: `148,990,000원`
- 상품별 합계: `148,990,000원`
- 월별 합계: `148,990,000원`
- 고객별 합계: `148,990,000원`
- 차이 여부: 차이 없음

![총합 검증](images/step05_total_check.png)

### 나의 해석과 판단
총합 검증은 같은 completed 주문 범위를 끝까지 유지했는지 확인하기 위해 필요합니다.
원본 completed 데이터의 `line_total` 합계와 카테고리별, 상품별, 월별, 고객별 합계가 모두 일치했으므로 집계 과정에서 누락이나 중복이 발생하지 않았다고 판단했습니다.

만약 총합이 일치하지 않는다면 먼저 completed 필터가 모든 집계에 동일하게 적용되었는지 확인해야 합니다. 
다음으로 merge 과정에서 행이 중복되거나 누락되지 않았는지, groupby 기준에 결측값이 있어 합계에서 빠진 항목이 없는지, 주문 수를 계산할 때 `count()`가 아니라 `order_id.nunique()`를 사용했는지 확인해야 합니다.

## 6. LLM pandas 코드 검증
- LLM Prompt 요약: completed 주문 기준으로 카테고리별 `total_sales`를 계산하고, merge와 총합 검증을 포함한 pandas 코드를 요청
- 제안 코드 요약: CSV 로드, merge, completed 필터링, `line_total` 생성, groupby 집계 코드 제안
- 실제 컬럼/범위와 맞지 않은 부분: 실제 데이터의 상태값, key 중복 여부, merge 관계 검증은 직접 확인해야 했음
- 수정한 내용: `validate="many_to_one"`, `indicator=True`, `order_id.nunique()`, completed 범위 총합 검증을 명시적으로 추가
- 최종 판단: 수정 후 사용

![LLM 코드 검증](images/step06_llm_validation.png)

### 나의 해석과 판단
LLM 코드가 실행된다는 것만으로 분석이 맞다고 볼 수는 없습니다. 
코드가 실제 컬럼명과 다른 이름을 사용하거나, 주문 상태값을 잘못 가정하거나, merge 관계를 검증하지 않아도 문법적으로는 실행될 수 있기 때문입니다. 특히 이번 과제에서는 completed 주문 범위를 끝까지 유지하는 것이 중요하므로, LLM이 제안한 코드도 실제 데이터 구조와 총합 검증을 통해 직접 확인해야 합니다.

## 7. Chapter 04 최종 인사이트
### 가장 의미 있다고 생각한 결과 2가지
1. completed 주문 기준 금액이 가장 큰 카테고리는 `스포츠`이며, `total_sales`는 `31,743,000원`입니다.
2. completed 주문 전체 `line_total` 합계와 카테고리별, 상품별, 월별, 고객별 합계가 모두 `148,990,000원`으로 일치했습니다.

### 그 결과를 뒷받침하는 수치/표
카테고리별 집계에서 `스포츠` 카테고리가 가장 높은 `total_sales`를 기록했습니다. 
또한 원본 completed 데이터의 `line_total` 합계와 여러 기준의 groupby 합계가 모두 동일하게 나타나 분석 범위가 일관되게 유지되었음을 확인했습니다.

### 추가로 확인하고 싶은 질문
스포츠 카테고리의 금액이 높은 이유가 판매 수량 때문인지, 단가 때문인지 추가로 확인하고 싶습니다. 
또한 특정 상품이나 특정 고객이 스포츠 카테고리의 금액을 크게 끌어올렸는지도 분석해보고 싶습니다. 
할인, 환불, 원가 데이터가 있다면 실제 수익성 기준으로도 다시 비교해볼 수 있습니다.

### 현재 결과의 한계
이번 결과는 completed 주문의 주문 상세 금액 합계만을 기준으로 합니다. 
따라서 실제 매출, 순매출, 이익, 고객 선호도를 직접 의미하지 않습니다. 
데이터 기간도 제한되어 있으므로 장기적인 추세나 시즌성을 단정하기 어렵습니다.

## 최종 제출 체크
- [x] 핵심 셀 Output이 남아 있습니다.
- [x] merge와 총합 검증 Evidence가 있습니다.
- [x] 결과 관찰과 해석이 구분되어 있습니다.
- [x] LLM 코드를 검증했습니다.
- [x] 개인정보/Secret이 없습니다.
- [x] `chapter04/chapter04.ipynb`가 GitHub에서 정상 표시됩니다.
- [x] 최종 Notebook 파일 URL을 제출합니다.